# 15 · Check R (part 1): rebuild the data from scratch with this folder's code only

Amendment A4 in `PLAN.md` (written before running; information only). Into a **fresh, empty** data folder, this
notebook downloads the inputs and rebuilds a sample of tiles and all NASA POWER daily data using only the code in
`askmap` (no code or data from the root project), then compares them with the files every result in this folder
was computed from.

- Downloads: USPVDB solar farms, Census state outlines, North Dakota's National Wetlands Inventory (1.5 GB).
- Wetlands: the NWI raster of all six North Dakota tiles, rebuilt from the freshly downloaded state file.
- NASA POWER: daily data 2001–2025 for both boxes.
- Comparisons: exact equality of every NWI raster and every POWER value; SHA-256 of downloads.

The Earth Engine layers (AlphaEarth, CDL) are rebuilt in part 2 (notebook 15b), which needs a current Earth Engine
login.

In [1]:
import json
import time

import numpy as np
import xarray as xr

from askmap.config import PROJECT_ROOT, data_root, load_config, load_env
from askmap.data import fetch_file
from askmap.manifest import Manifest, sha256_of
from askmap.nwi import nwi_raster, read_nwi_tile
from askmap.power import fetch_daily
from askmap.tiles import area_tiles

cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
OLD = data_root()  # the data every recorded result was computed from
NEW = OLD.parent / "ask_the_map_rebuild_check"  # fresh folder for this check
if NEW.exists() and any(NEW.iterdir()):
    raise RuntimeError(f"{NEW} is not empty; this check must start from an empty folder")
NEW.mkdir(parents=True, exist_ok=True)
RES = PROJECT_ROOT / "outputs" / "results"
load_env()
YEAR, P, U = cfg["land_year"], cfg["power"], cfg["uspvdb"]
T_START = time.time()

## Downloads

In [2]:
raw_new = Manifest(NEW / "raw")
t0 = time.time()
pv_zip = fetch_file(U["url"], NEW / "raw" / "uspvdbGeoJSON.zip", raw_new)
st_zip = fetch_file(P["states_url"], NEW / "raw" / "cb_2023_us_state_20m.zip", raw_new)
nd = cfg["areas"]["north_dakota"]
nwi_zip = fetch_file(nd["nwi_url"], NEW / nd["nwi_zip"], raw_new, timeout=3600)
print(f"downloads in {time.time() - t0:.0f}s")
old_raw = json.loads((OLD / "raw" / "manifest.json").read_text())
# September's copy, in the root project's data folder (read only, for this comparison)
SEPT_NWI = OLD.parent / "raw" / "wetlands" / "ND_geodatabase_wetlands.zip"
downloads = {
    "uspvdb_same_sha256": raw_new.entries[pv_zip.name]["sha256"] == old_raw[pv_zip.name]["sha256"],
    "states_same_sha256": raw_new.entries[st_zip.name]["sha256"] == old_raw[st_zip.name]["sha256"],
    "nwi_bytes": raw_new.entries[nwi_zip.name]["bytes"],
}
if SEPT_NWI.exists():
    downloads["nwi_same_sha256_as_september"] = raw_new.entries[nwi_zip.name]["sha256"] == sha256_of(SEPT_NWI)
else:
    downloads["nwi_same_sha256_as_september"] = None
    print(f"September's NWI copy not found at {SEPT_NWI}; the NWI file is compared through the tiles only")
downloads

downloads in 150s


{'uspvdb_same_sha256': True,
 'states_same_sha256': True,
 'nwi_bytes': 1503003179,
 'nwi_same_sha256_as_september': False}

## Wetland rasters (North Dakota tiles) from the fresh NWI download

In [3]:
rows = []
for name, (tile, _role) in area_tiles(nd, cfg["grid"]).items():
    t0 = time.time()
    new = nwi_raster(read_nwi_tile(f"zip://{nwi_zip}!{nd['nwi_gdb']}", nd["nwi_layer"], tile), tile)
    old = np.load(OLD / "tiles" / f"north_dakota_{name}.npz")["nwi"]
    row = {"tile": name, "seconds": round(time.time() - t0, 1), "identical": bool(np.array_equal(old, new))}
    if not row["identical"]:
        row["share_of_pixels_different"] = float((old != new).mean())
        row["wetland_share_old_new"] = [float((old == 1).mean()), float((new == 1).mean())]
        row["water_share_old_new"] = [float((old == 2).mean()), float((new == 2).mean())]
    rows.append(row)
    print(row)

{'tile': 'Stutsman', 'seconds': 7.9, 'identical': True}


{'tile': 'Barnes', 'seconds': 7.2, 'identical': True}


{'tile': 'Wells', 'seconds': 7.1, 'identical': True}


{'tile': 'Kidder', 'seconds': 7.4, 'identical': True}


{'tile': 'Foster', 'seconds': 6.8, 'identical': True}


{'tile': 'Eddy', 'seconds': 7.0, 'identical': True}


## NASA POWER daily data (both boxes)

In [4]:
power_rows = []
for box, b in P["boxes"].items():
    t0 = time.time()
    new = fetch_daily(P, b, NEW / "power" / f"universe_{box}_daily.nc")
    old = xr.open_dataset(OLD / "power" / f"universe_{box}_daily.nc", engine="h5netcdf").load()
    same_grid = all(np.array_equal(old[c].values, new[c].values) for c in ("time", "lat", "lon"))
    per_var = {v: bool(np.array_equal(old[v].values, new[v].values, equal_nan=True)) for v in old.data_vars}
    power_rows.append(
        {
            "box": box,
            "seconds": round(time.time() - t0),
            "same_grid": same_grid,
            "variables_identical": per_var,
        }
    )
    print(power_rows[-1])

POWER daily: {'lat': 28, 'lon': 21, 'time': 9131}; missing values {'T2M': 0, 'T2M_MAX': 0, 'T2M_MIN': 0, 'PRECTOTCORR': 0, 'GWETROOT': 0, 'WS50M': 0, 'RH2M': 0, 'ALLSKY_SFC_SW_DWN': 0}


{'box': 'plains', 'seconds': 49, 'same_grid': True, 'variables_identical': {'T2M': True, 'T2M_MAX': True, 'T2M_MIN': True, 'PRECTOTCORR': True, 'GWETROOT': True, 'WS50M': True, 'RH2M': True, 'ALLSKY_SFC_SW_DWN': True}}


POWER daily: {'lat': 20, 'lon': 16, 'time': 9131}; missing values {'T2M': 0, 'T2M_MAX': 0, 'T2M_MIN': 0, 'PRECTOTCORR': 0, 'GWETROOT': 538729, 'WS50M': 0, 'RH2M': 0, 'ALLSKY_SFC_SW_DWN': 0}


{'box': 'southeast', 'seconds': 38, 'same_grid': True, 'variables_identical': {'T2M': True, 'T2M_MAX': True, 'T2M_MIN': True, 'PRECTOTCORR': True, 'GWETROOT': True, 'WS50M': True, 'RH2M': True, 'ALLSKY_SFC_SW_DWN': True}}


## Verdict (information)

In [5]:
tiles_ok = all(r["identical"] for r in rows)
power_ok = all(r["same_grid"] and all(r["variables_identical"].values()) for r in power_rows)
check = {
    "downloads": downloads,
    "nwi_rasters": rows,
    "power": power_rows,
    "nwi_identical": tiles_ok,
    "power_identical": power_ok,
    "minutes": round((time.time() - T_START) / 60, 1),
    "fresh_folder_MB": round(sum(p.stat().st_size for p in NEW.rglob("*") if p.is_file()) / 1e6),
}
(RES / "15_rebuild_check.json").write_text(json.dumps(check, indent=1, default=str))
print(
    f"NWI rasters identical: {tiles_ok}; POWER identical: {power_ok}; {check['minutes']} min; "
    f"fresh folder {check['fresh_folder_MB']} MB"
)

NWI rasters identical: True; POWER identical: True; 4.7 min; fresh folder 1778 MB
